# Pré-processamento e Engenharia de Features

Este notebook implementa o pipeline de pré-processamento com as seguintes melhorias:

1. **Correção de Data Leakage**: Scaler é fitado apenas nos dados de treino
2. **Janelas Configuráveis**: Experimentos com W=3, W=7, W=15
3. **Modularização**: Uso de funções do módulo `src/features.py`

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.preprocessing import StandardScaler
import joblib

_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, 'src')):
        break
    _root = os.path.dirname(_root)
sys.path.insert(0, os.path.join(_root, 'src'))
from features import preparar_features, criar_features_multi_horizonte

## 1. Carregamento dos Dados

In [2]:
df = pd.read_parquet("../data/processed/vendas_supermercado.parquet", engine="pyarrow")
print("Shape original:", df.shape)

Shape original: (3415725, 20)


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3415725 entries, 0 to 3415724
Data columns (total 20 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   Tipo_Registro             str           
 1   Data                      datetime64[us]
 2   Hora                      str           
 3   Cod_Produto               str           
 4   Cod_Secao                 category      
 5   Cod_Loja                  category      
 6   Quantidade                float64       
 7   Valor_Unitario            float64       
 8   Desconto                  float64       
 9   Preco_Custo               float64       
 10  Acréscimos                float64       
 11  Preço praticado           int64         
 12  Total                     float64       
 13  Preço de Venda            float64       
 14  Tipo de bonificação       str           
 15  Fator                     str           
 16  Identificação Consumidor  str           
 17  Operador           

## 2. Seleção do Top SKU

In [4]:
produto_top = (
    df.groupby("Cod_Produto")["Quantidade"]
      .sum()
      .idxmax()
)

print("Produto mais vendido:", produto_top)

Produto mais vendido: 00000000022134


In [5]:
df_top = df[df["Cod_Produto"] == produto_top].copy()

In [6]:
df_top["Data"] = pd.to_datetime(df_top["Data"])

In [7]:
df_daily = (
    df_top.groupby("Data")["Quantidade"]
    .sum()
    .reset_index()
    .sort_values("Data")
)

df_daily

,Data,Quantidade
0,2023-01-03,451.0
1,2023-01-04,428.0
2,2023-01-05,350.0
3,2023-01-06,403.0
4,2023-01-07,234.0
...,...,...
895,2025-12-26,377.0
896,2025-12-27,244.0
897,2025-12-29,385.0
898,2025-12-30,644.0


In [8]:
df_daily.describe()

,Data,Quantidade
count,900,900.000000
mean,2024-07-06 18:17:36,417.432222
min,2023-01-03 00:00:00,137.000000
25%,2023-10-04 18:00:00,356.750000
50%,2024-07-09 12:00:00,413.000000
75%,2025-04-09 06:00:00,467.000000
max,2025-12-31 00:00:00,886.000000
std,NaN,103.323174


## 3. Engenharia de Features com Janelas Configuráveis

### Experimento 1: Configuração Padrão (lag=7, W=7)

In [9]:
df_features = preparar_features(df_daily, max_lag=7, janela_rolling=7)
print("Shape após feature engineering:", df_features.shape)
df_features.head()

Shape após feature engineering: (893, 15)


,Data,Quantidade,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,media_7,std_7,dia_semana,mes,dia_mes,fim_de_semana
7,2023-01-11,421.0,337.0,436.0,234.0,403.0,350.0,428.0,451.0,377.000000,72.276585,2,1,11,0
8,2023-01-12,372.0,421.0,337.0,436.0,234.0,403.0,350.0,428.0,372.714286,68.116846,3,1,12,0
9,2023-01-13,599.0,372.0,421.0,337.0,436.0,234.0,403.0,350.0,364.714286,110.796682,4,1,13,0
10,2023-01-14,307.0,599.0,372.0,421.0,337.0,436.0,234.0,403.0,400.285714,116.213678,5,1,14,1
11,2023-01-16,479.0,307.0,599.0,372.0,421.0,337.0,436.0,234.0,386.571429,98.084754,0,1,16,0


## 4. Divisão Temporal (ANTES do Scaling)

**IMPORTANTE**: Esta é a correção do data leakage. Dividimos os dados ANTES de aplicar o scaler.

In [10]:
split_index = int(len(df_features) * 0.8)

df_train = df_features.iloc[:split_index].copy()
df_test = df_features.iloc[split_index:].copy()

print(f"Treino: {len(df_train)} registros")
print(f"Teste: {len(df_test)} registros")

Treino: 714 registros
Teste: 179 registros


## 5. Padronização (Fit apenas no Treino)

**Correção Crítica**: O scaler é fitado APENAS nos dados de treino, evitando vazamento de informação futura.

In [11]:
colunas_numericas = df_train.columns.drop("Data")

scaler = StandardScaler()

df_train[colunas_numericas] = scaler.fit_transform(df_train[colunas_numericas])
df_test[colunas_numericas] = scaler.transform(df_test[colunas_numericas])

print("Scaler fitado apenas nos dados de treino!")

Scaler fitado apenas nos dados de treino!


In [12]:
df_train.head()

,Data,Quantidade,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,media_7,std_7,dia_semana,mes,dia_mes,fim_de_semana
7,2023-01-11,-0.026647,-0.827015,0.116879,-1.803994,-0.196450,-0.699508,0.044924,0.263856,-1.412318,-0.908023,-0.291551,-1.460784,-0.564542,-0.444957
8,2023-01-12,-0.494415,-0.025511,-0.827857,0.119253,-1.806946,-0.193074,-0.700578,0.044016,-1.542519,-1.033915,0.294836,-1.460784,-0.450257,-0.444957
9,2023-01-13,1.672593,-0.493055,-0.026263,-0.823328,0.118025,-1.807930,-0.194019,-0.701527,-1.785561,0.257767,0.881222,-1.460784,-0.335972,-0.444957
10,2023-01-14,-1.114924,1.672914,-0.493859,-0.023563,-0.825401,0.122252,-1.809272,-0.194940,-0.704891,0.421710,1.467608,-1.460784,-0.221687,2.247409
11,2023-01-16,0.527038,-1.113267,1.672353,-0.490093,-0.024918,-0.823728,0.121386,-1.810282,-1.121535,-0.126953,-1.464323,-1.460784,0.006883,-0.444957


## 6. Salvamento dos Dados Processados

In [13]:
df_train.to_parquet("../data/processed/train_features.parquet", engine="pyarrow")
df_test.to_parquet("../data/processed/test_features.parquet", engine="pyarrow")
print("Dados de treino e teste salvos!")

Dados de treino e teste salvos!


In [14]:
joblib.dump(scaler, '../models/scaler_quantidade.pkl')
print("Scaler salvo em ../models/scaler_quantidade.pkl")

Scaler salvo em ../models/scaler_quantidade.pkl


## 7. Experimentos com Diferentes Janelas

### Matriz de Configurações

In [15]:
configuracoes = [
    (3, 3),    # Lags curtos, janela curta
    (7, 7),    # Configuração padrão
    (15, 15),  # Lags longos, janela longa
    (7, 3),    # Lags padrão, janela curta
    (7, 15),   # Lags padrão, janela longa
    (3, 7)     # Lags curtos, janela padrão
]

from features import preparar_features_multiplas_janelas
resultados_configs = preparar_features_multiplas_janelas(df_daily, configuracoes)

for nome, df_config in resultados_configs.items():
    print(f"{nome}: {df_config.shape}")

Configuração lag3_w3: (897, 11)
Configuração lag7_w7: (893, 15)
Configuração lag15_w15: (885, 23)
Configuração lag7_w3: (893, 15)
Configuração lag7_w15: (885, 15)
Configuração lag3_w7: (893, 11)
lag3_w3: (897, 11)
lag7_w7: (893, 15)
lag15_w15: (885, 23)
lag7_w3: (893, 15)
lag7_w15: (885, 15)
lag3_w7: (893, 11)


### Salvar Todas as Configurações

In [16]:
for nome, df_config in resultados_configs.items():
    split_idx = int(len(df_config) * 0.8)
    df_t = df_config.iloc[:split_idx].copy()
    df_te = df_config.iloc[split_idx:].copy()
    
    cols = df_t.columns.drop("Data")
    scaler_temp = StandardScaler()
    df_t[cols] = scaler_temp.fit_transform(df_t[cols])
    df_te[cols] = scaler_temp.transform(df_te[cols])
    
    df_t.to_parquet(f"../data/processed/train_{nome}.parquet", engine="pyarrow")
    df_te.to_parquet(f"../data/processed/test_{nome}.parquet", engine="pyarrow")

print("Todas as configurações salvas!")

Todas as configurações salvas!


## 8. Preparação para MIMO (Multi-Input Multi-Output)

Criação de targets para múltiplos dias à frente (horizonte=7).

In [17]:
df_mimo = criar_features_multi_horizonte(df_daily, horizonte=7, max_lag=7, janela_rolling=7)
print("Shape MIMO:", df_mimo.shape)
df_mimo.head()

Shape MIMO: (886, 22)


,Data,Quantidade,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,media_7,...,mes,dia_mes,fim_de_semana,target_d1,target_d2,target_d3,target_d4,target_d5,target_d6,target_d7
7,2023-01-11,421.0,337.0,436.0,234.0,403.0,350.0,428.0,451.0,377.000000,...,1,11,0,372.0,599.0,307.0,479.0,429.0,457.0,387.0
8,2023-01-12,372.0,421.0,337.0,436.0,234.0,403.0,350.0,428.0,372.714286,...,1,12,0,599.0,307.0,479.0,429.0,457.0,387.0,467.0
9,2023-01-13,599.0,372.0,421.0,337.0,436.0,234.0,403.0,350.0,364.714286,...,1,13,0,307.0,479.0,429.0,457.0,387.0,467.0,300.0
10,2023-01-14,307.0,599.0,372.0,421.0,337.0,436.0,234.0,403.0,400.285714,...,1,14,1,479.0,429.0,457.0,387.0,467.0,300.0,380.0
11,2023-01-16,479.0,307.0,599.0,372.0,421.0,337.0,436.0,234.0,386.571429,...,1,16,0,429.0,457.0,387.0,467.0,300.0,380.0,349.0


In [18]:
df_mimo.to_parquet("../data/processed/mimo_features.parquet", engine="pyarrow")
print("Dados MIMO salvos!")

Dados MIMO salvos!


## Resumo

Neste notebook:

1. ✓ Corrigido data leakage do scaler (fit apenas no treino)
2. ✓ Criadas múltiplas configurações de janelas (W=3, 7, 15)
3. ✓ Preparados dados para abordagem MIMO
4. ✓ Salvos dados de treino e teste separadamente

**Próximo passo**: Executar `03_modeling.ipynb` para treinar os modelos com essas configurações.